# Reprodução didática de Particle Swarm Optimization (PSO)

Experimento usado no seminário TP558. O objetivo é visualizar convergência, trajetória e estabilidade para diferentes configurações de parâmetros.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(X):
    X = np.asarray(X)
    return np.sum(X**2 + 10*(1-np.cos(2*np.pi*X)), axis=-1)


In [ ]:
def run_pso(seed=2, w=0.7298, c1=1.49618, c2=1.49618, iters=80, n=30, decay=None, store=False):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-5.12, 5.12, (n, 2))
    v = rng.uniform(-1, 1, (n, 2))
    p = x.copy(); pfit = f(x)
    gi = np.argmin(pfit); g = p[gi].copy(); gfit = float(pfit[gi])
    best = [gfit]; mean_v = [np.mean(np.linalg.norm(v, axis=1))]
    hist = [x.copy()] if store else None
    for t in range(iters):
        wt = decay(t, iters) if decay else w
        r1 = rng.random((n,2)); r2 = rng.random((n,2))
        v = wt*v + c1*r1*(p-x) + c2*r2*(g-x)
        x = x + v
        vals = f(x)
        better = vals < pfit
        p[better] = x[better]; pfit[better] = vals[better]
        gi = np.argmin(pfit)
        if pfit[gi] < gfit:
            gfit = float(pfit[gi]); g = p[gi].copy()
        best.append(gfit); mean_v.append(np.mean(np.linalg.norm(v, axis=1)))
        if store: hist.append(x.copy())
    return dict(best=np.asarray(best), mean_v=np.asarray(mean_v), g=g, gfit=gfit, hist=hist)


In [ ]:
A = run_pso(w=0.7298, c1=1.49618, c2=1.49618, store=True)
B = run_pso(w=0.9, c1=2.0, c2=2.0, decay=lambda t,T: 0.9 - 0.5*t/(T-1))
C = run_pso(w=1.2, c1=2.5, c2=2.5)
print("Melhor fitness final")
print("Constrição:", A["gfit"])
print("Inércia decrescente:", B["gfit"])
print("Configuração agressiva:", C["gfit"])


In [ ]:
plt.figure(figsize=(9,4.8))
plt.semilogy(A['best']+1e-16, label='Constrição')
plt.semilogy(B['best']+1e-16, label='Inércia decrescente')
plt.semilogy(C['best']+1e-16, label='Configuração agressiva')
plt.xlabel('Iteração'); plt.ylabel('Melhor fitness conhecido')
plt.grid(alpha=.2, which='both'); plt.legend(); plt.show()


In [ ]:
xx=np.linspace(-5.12,5.12,400); yy=np.linspace(-5.12,5.12,400)
X,Y=np.meshgrid(xx,yy); Z=f(np.dstack([X,Y]))
plt.figure(figsize=(8,6))
plt.contour(X,Y,Z,levels=[1,5,10,20,40,80,120])
for idx in [0,3,7,11,18,24]:
    P=np.array([h[idx] for h in A['hist']])
    plt.plot(P[:,0],P[:,1],alpha=.6)
plt.scatter(0,0,marker='*',s=120,label='Ótimo global')
plt.xlim(-5.2,5.2); plt.ylim(-5.2,5.2)
plt.xlabel('$x_1$'); plt.ylabel('$x_2$'); plt.legend(); plt.show()
